# Stepageddon - Train Step Chart Model

Train a neural network to generate DDR step charts from audio.

**Prerequisites:**
1. Run `prepare_data.py` locally to preprocess your charts
2. Zip the training data: `cd backend/ml && zip -r training_data.zip training_data/`
3. Upload `training_data.zip` directly to Colab when prompted (cell below)

**Runtime:** Select GPU (T4) under Runtime > Change runtime type

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q librosa numpy torch simfile

In [ ]:
# Pull the source from Google Drive into /content so imports resolve.
#
# Expected Drive layout (adjust SRC if yours differs):
#   /content/drive/MyDrive/stepageddon/backend/
#       ml/         (model.py, dataset.py, train.py, prepare_data.py, ...)
#       services/   (sm_parser.py used by prepare_data; safe to skip if you
#                    only need ml.train at runtime, but harmless to copy)
#       modules/    (step_generator schemas — required by inference.py)
#
# We copy these to /content/{ml,services,modules}. Cell 7 adds /content to
# sys.path so `import ml.train`, `import services.sm_parser`, and
# `import modules.step_generator...` all work without any package install.
import os, shutil
from pathlib import Path

SRC = Path('/content/drive/MyDrive/stepageddon')
DST = Path('/content')

assert SRC.exists(), f'Source not found: {SRC}. Adjust SRC to your Drive layout.'

for pkg in ('ml', 'services', 'modules'):
    src_pkg = SRC / pkg
    dst_pkg = DST / pkg
    if not src_pkg.exists():
        print(f'  skip {pkg}/ (not in Drive)')
        continue
    if dst_pkg.exists():
        shutil.rmtree(dst_pkg)
    # dirs_exist_ok unused since we cleaned above; copytree handles symlinks fine.
    shutil.copytree(src_pkg, dst_pkg)
    # Make sure each top-level dir is a real Python package.
    init_file = dst_pkg / '__init__.py'
    if not init_file.exists():
        init_file.touch()
    print(f'  copied {pkg}/  ({sum(1 for _ in dst_pkg.rglob("*.py"))} .py files)')

# Sanity-check the ml/ contents we actually need to train.
for needed in ('ml/model.py', 'ml/dataset.py', 'ml/train.py', 'ml/prepare_data.py'):
    assert (DST / needed).exists(), f'Missing {needed} after copy'
print('\nAll required modules in place.')
!ls /content/ml/


In [ ]:
# Upload and extract training data
# Pick ONE option below and comment out the others

import zipfile, os
ZIP_PATH = '/content/drive/MyDrive/stepageddon/training_data.zip'  # adjust path if needed
print(f'Copying and extracting from Drive...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')
print('Done!')

!ls /content/training_data/ | head -20

In [ ]:
# Verify data is accessible (manifest format_version=2)
import json
from pathlib import Path

DATA_DIR = Path('/content/training_data')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/stepageddon/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = DATA_DIR / 'manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)

entries = manifest['entries']
print(f'Total training examples: {len(entries)}')
print(f'Mel whitening stats: mean={manifest["mel_mean"]:.4f} std={manifest["mel_std"]:.4f}')

# Show difficulty distribution
from collections import Counter
diff_counts = Counter(e['difficulty'] for e in entries)
print(f'Difficulty distribution: {dict(diff_counts)}')

# Check a sample file (one npz per song; labels live under labels_<diff>)
import numpy as np
sample_entry = entries[0]
sample = np.load(DATA_DIR / sample_entry['filename'])
labels = sample[sample_entry['labels_key']]
print(f"\nSample: {sample_entry['song_title']} ({sample_entry['difficulty']})")
print(f'  Mel shape: {sample["mel"].shape}')
print(f'  Labels shape: {labels.shape}')


In [ ]:
# Add to Python path
import sys
sys.path.insert(0, '/content')


In [ ]:
# Build dataloaders, model, optimizer, loss using train.py helpers.
# Keep the Colab-specific config here; delegate all training logic to ml.train
# so the notebook and the CLI always run the exact same code path.
from types import SimpleNamespace
import torch
from torch.amp import GradScaler
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

from ml.train import (
    build_dataloaders, build_model, build_optimizer_and_scheduler,
    build_loss, seed_everything,
)

# Hyperparameters — keep in sync with ml.train.build_argparser defaults.
#
# Arrow-aware variant: replaces the old single-onset head with a 4-way
# per-arrow head conditioned on the previous emitted arrow vector. The
# model now learns P(arrow_t | audio_t, arrow_{prev}) directly, so arrow
# choice is no longer outsourced to FootStateArrowAssigner. The diversity
# regularizer + per-arrow pos_weight prevent collapse to a single arrow.
#
# REQUIRES re-running prepare_data.py (FORMAT_VERSION bumped to 5 — adds
# arrow_labels_<diff> per chart in the npz).
args = SimpleNamespace(
    data_dir=str(DATA_DIR),
    checkpoint_dir=str(CHECKPOINT_DIR),
    epochs=80,
    batch_size=32,
    lr=3e-4,
    hidden_dim=256,
    n_heads=8,
    n_layers=4,
    tcn_dilations='1,2,4,8,16,32',
    onset_head_layers=3,
    chunk_frames=500,        # 5s at ~100fps
    val_split=0.1,
    weight_decay=0.01,
    warmup_epochs=2.0,
    type_weight=4.0,
    duration_weight=10.0,
    type_weight_smoothing=0.5,
    type_weight_cap=10.0,
    type_focal_gamma=0.0,
    pos_weight=None,                # per-arrow defaults from arrow_priors
    focal_gamma=0.5,
    jump_sample_boost=8.0,
    hold_sample_boost=16.0,
    p_rare=0.2,
    p_density_swap=0.5,
    mixup_p=0.3,
    mixup_alpha=0.4,
    head_lr_mult=2.0,
    beat_weight=0.3,
    diversity_weight=0.2,           # per-chunk arrow-distribution KL reg
    prev_arrow_dropout=0.1,         # zero prev_arrow this fraction of frames
    plateau_patience=4,
    plateau_factor=0.5,
    dropout=0.1,
    num_workers=2,
    ema_decay=0.999,
    seed=42,
    tol_frames=3,            # ~30ms at 100fps
    log_interval=50,
)

seed_everything(args.seed)
device = torch.device('cuda')

built = build_dataloaders(args)
model = build_model(
    args, built.arrow_priors, built.type_prior, device,
    hold_duration_median=built.hold_duration_median,
    beat_prior=built.beat_prior,
)
criterion = build_loss(built.arrow_priors, built.type_prior, args).to(device)
optimizer, scheduler = build_optimizer_and_scheduler(
    model, args, steps_per_epoch=max(1, len(built.train_loader)),
)
scaler = GradScaler('cuda')
ema_model = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(args.ema_decay))

print(f'Train entries: {len(built.train_idx)}  Val entries: {len(built.val_idx)}')
print(f'Empirical onset prior: {built.onset_prior:.6f}')
print(f'Empirical arrow priors [L,D,U,R]: {built.arrow_priors.tolist()}')
print(f'Empirical type prior (tap/jump/hold): {built.type_prior.tolist()}')
print(f'Empirical hold-duration median: {built.hold_duration_median:.3f} beats')
print(f'Empirical beat prior: {built.beat_prior:.6f}')
print(f'Empirical density/difficulty: {built.default_density_by_id.tolist()}')


In [ ]:
# Training loop — calls into ml.train.train_one_epoch / validate.
# Auto-resumes from last_model.pt on Drive if it exists, so re-running this
# cell after a Colab disconnect picks up where it left off instead of
# restarting from epoch 0.
import time
from ml.train import train_one_epoch, validate, save_checkpoint

history = {'train_loss': [], 'val_loss': [], 'tol_f1': [], 'macro_f1': [], 'composite': []}
best_composite = -1.0
start_epoch = 0


def composite_score(metrics):
    """Headline checkpoint score: weighted blend of onset timing and
    calibrated type macro-F1. Mirrors ml.train.main()._composite."""
    tol = float(metrics.get('tol_f1', 0.0))
    cal = float(metrics.get('cal_macro_f1', metrics.get('type_macro_f1', 0.0)))
    return 0.6 * tol + 0.4 * cal


# Resume from last checkpoint if present (survives Colab disconnects).
last_ckpt_path = CHECKPOINT_DIR / 'last_model.pt'
if last_ckpt_path.exists():
    print(f'Resuming from {last_ckpt_path}')
    ckpt = torch.load(last_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    if ckpt.get('ema_state_dict') is not None:
        ema_model.module.load_state_dict(ckpt['ema_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_composite = composite_score(ckpt.get('metrics', {}))
    print(f'  -> resumed at epoch {start_epoch}, best composite={best_composite:.3f}')
else:
    print('No checkpoint found — starting fresh from epoch 0.')

for epoch in range(start_epoch, args.epochs):
    t0 = time.time()
    train_metrics = train_one_epoch(
        model, built.train_loader, criterion, optimizer, scheduler,
        scaler, device, ema_model=ema_model,
        log_interval=args.log_interval,
        prev_arrow_dropout=args.prev_arrow_dropout,
    )
    val_metrics = validate(
        ema_model.module, built.val_loader, criterion, device,
        tol_frames=args.tol_frames,
    )

    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]['lr']
    composite = composite_score(val_metrics)
    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['tol_f1'].append(val_metrics['tol_f1'])
    history['macro_f1'].append(val_metrics['type_macro_f1'])
    history['composite'].append(composite)

    print(
        f"Epoch {epoch+1:3d}/{args.epochs} | "
        f"train={train_metrics['loss']:.4f} "
        f"(arrow={train_metrics['arrow_loss']:.4f} type={train_metrics['type_loss']:.4f} "
        f"dur={train_metrics['duration_loss']:.4f} beat={train_metrics['beat_loss']:.4f} "
        f"div={train_metrics['diversity_loss']:.4f}) "
        f"| val={val_metrics['loss']:.4f} "
        f"tol_f1={val_metrics['tol_f1']:.3f} "
        f"(P={val_metrics['tol_precision']:.3f} R={val_metrics['tol_recall']:.3f} thr={val_metrics['best_thr']:.2f}) "
        f"arrow_F1[L/D/U/R]={val_metrics['arrow_f1_L']:.2f}/"
        f"{val_metrics['arrow_f1_D']:.2f}/{val_metrics['arrow_f1_U']:.2f}/"
        f"{val_metrics['arrow_f1_R']:.2f} "
        f"(macro={val_metrics['arrow_f1_macro']:.3f}) "
        f"arrow_kl={val_metrics['arrow_marginal_kl']:.3f} "
        f"stream_coh={val_metrics['stream_coherence']:.3f} "
        f"| type_F1[t/j/h]={val_metrics['tap_f1']:.3f}/"
        f"{val_metrics['jump_f1']:.3f}/{val_metrics['hold_f1']:.3f} "
        f"(macro={val_metrics['type_macro_f1']:.3f}) "
        f"cal_F1[t/j/h]={val_metrics['cal_tap_f1']:.3f}/"
        f"{val_metrics['cal_jump_f1']:.3f}/{val_metrics['cal_hold_f1']:.3f} "
        f"(cal_macro={val_metrics['cal_macro_f1']:.3f} "
        f"jb={val_metrics['cal_jump_bias']:+.2f} hb={val_metrics['cal_hold_bias']:+.2f}) "
        f"dur_mae={val_metrics['dur_mae_beats']:.3f}b "
        f"| composite={composite:.3f} "
        f"| lr={lr:.2e} | {elapsed:.1f}s",
        flush=True,
    )

    save_checkpoint(
        CHECKPOINT_DIR / 'last_model.pt', epoch, model, ema_model,
        optimizer, scheduler, scaler, val_metrics,
        built.default_density_by_id,
        built.full_dataset.mel_mean, built.full_dataset.mel_std,
        built.type_prior, built.arrow_priors,
        args,
    )
    if composite > best_composite:
        best_composite = composite
        save_checkpoint(
            CHECKPOINT_DIR / 'best_model.pt', epoch, model, ema_model,
            optimizer, scheduler, scaler, val_metrics,
            built.default_density_by_id,
            built.full_dataset.mel_mean, built.full_dataset.mel_std,
            built.type_prior, built.arrow_priors,
            args,
        )
        print(f"  -> New best composite={best_composite:.3f} "
              f"(tol_f1={val_metrics['tol_f1']:.3f} "
              f"cal_macro={val_metrics['cal_macro_f1']:.3f})", flush=True)

print(f'\nTraining complete! Best composite={best_composite:.3f}')
print(f'Best model saved at: {CHECKPOINT_DIR}/best_model.pt')


## After Training

1. Download `best_model.pt` from Google Drive (`stepageddon/checkpoints/best_model.pt`)
2. Place it in `backend/ml/checkpoints/best_model.pt`
3. Set `USE_ML_GENERATION=true` in `backend/.env`
4. Restart the backend server